# Tuples, Sets & Nested Data: records, unique values and data inside data

**▶ 01 Core Python** · 02 Pandas · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · 07 Model Prep · 08 Case Studies · 09 Deployment

`01_Python_Fundamentals/02_tuples_sets_and_nested_data.ipynb`

---

### In one paragraph (no jargon)

Real business data is rarely one flat row. A quarterly sales report is a *list of lists*; an employee record is a *dictionary inside a dictionary*. This notebook teaches you to walk into those structures and pull out exactly the number you need. Along the way you meet two more containers: a **tuple** (a list that can't be edited, and is perfect for a fixed record like `('Delhi', 250000)`) and a **set** (a bag of unique values with no duplicates and no order, perfect for 'which customers do both branches share?').

### After this notebook you can

- Reach into nested lists and nested dictionaries with chained brackets
- Flatten a nested list into a flat one, three different ways
- Unpack a tuple into separate named variables in one line
- Use set operations (union, intersection, difference) to answer overlap questions instantly


### What's inside

1. Part 1: Nested lists (Q1-Q3)
2. ⚡ Three ways to flatten · namedtuple
3. Part 2: Nested dictionaries (Q4-Q6)
4. Part 3: Tuples (Q7-Q8)
5. Part 4: Sets (Q9-Q10)
6. ⚡ Set algebra for churn analysis
7. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory, so **every cell below still runs**,
> which is handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

### Jargon buster

| Word you'll see | What it actually means |
|---|---|
| **variable** | A labelled box that holds a value. `price = 250` puts 250 in a box called `price`. |
| **list** `[ ]` | An ordered shopping list of values. You can add, remove and reorder items. |
| **tuple** `( )` | Like a list, but frozen: once made it cannot be changed. Good for fixed records. |
| **dictionary** `{key: value}` | A labelled lookup table, like a contacts app: look up "Ravi", get his number. |
| **index** | The position number of an item. Python counts from **0**, so the 1st item is at index 0. |
| **function** | A named recipe you can re-run with different ingredients. |
| **argument / parameter** | The ingredients you hand to a function. |
| **return** | The answer a function hands back to you. |
| **iterate / loop** | Do the same thing once for every item in a list. |
| **boolean** | A yes/no value: `True` or `False`. |
| **string** `'text'` | Text. Always wrapped in quotes. |
| **f-string** `f"{x}"` | A sentence with live values slotted in: the report-writing tool. |

In [1]:
# =============================================================================
# SETUP — run this cell first. Everything below depends only on this.
# =============================================================================
# WHAT: these are Python's own built-in toolboxes; nothing needs installing.
import math                     # square roots, rounding, constants
import statistics               # mean / median without pandas
from collections import Counter, defaultdict   # counting and grouping helpers
from functools import reduce    # folds a whole list down to one value

# -----------------------------------------------------------------------------
# Exam-safe replacement for input()
# -----------------------------------------------------------------------------
# WHY: a notebook that calls input() STOPS and waits for typing. If the examiner
#      clicks "Run All", it hangs. `ask()` behaves exactly like input() but plays
#      back a scripted answer instead, so the notebook always runs end-to-end.
# 🔧 CHANGE THIS: set INTERACTIVE = True if you WANT to type answers yourself.
INTERACTIVE = False
_scripted_answers = iter(['42', '7', '3', '15', '100', '5', '2', '9', '1', '0'])

def ask(prompt=''):
    """Stand-in for input() that never blocks a Run All."""
    if INTERACTIVE:
        return input(prompt)
    value = next(_scripted_answers, '0')
    print(f"{prompt}{value}    <- simulated keyboard input")
    return value

print("Setup complete — you can now run any cell below in any order.")

Setup complete — you can now run any cell below in any order.


## Part 1: Nested lists

Read `sales[1][0]` **left to right**: go to the item at position 1 (the second inner list), then inside *that*, take position 0. Each pair of brackets steps one level deeper. If you get confused, print the intermediate step (`print(sales[1])`), and the next bracket becomes obvious.

**Q1. Given a nested list of quarterly sales `sales=[[1200,1300],[1400,1500],[1600,1700]]`, how can you access the first quarter of the second year??**

In [2]:
# Solution for Q1
sales = [[1200, 1300], [1400, 1500], [1600, 1700]]
first_quarter_first_month = sales[1][0]  # index year two, then grab its first quarter month
print(f"Year 2 Q1 first month sales: {first_quarter_first_month}")

Year 2 Q1 first month sales: 1400


**Q2. How can you update the value 1500 to 1550 in the nested list `sales=[[1200,1300],[1400,1500],[1600,1700]]`??**

In [3]:
# Solution for Q2
sales = [[1200, 1300], [1400, 1500], [1600, 1700]]
sales[1][1] = 1550  # second quarter, second month
print(f"Updated sales structure: {sales}")

Updated sales structure: [[1200, 1300], [1400, 1550], [1600, 1700]]


**Q3. How can you flatten the nested list `orders=[[101,102],[103,104],[105]]` into a single list??**

In [4]:
# Solution for Q3
orders = [[101, 102], [103, 104], [105]]
flat_orders = [order for batch in orders for order in batch]  # nested comprehension to flatten
print(f"Flattened order IDs: {flat_orders}")

Flattened order IDs: [101, 102, 103, 104, 105]


### ⚡ Beyond the syllabus: flattening a nested list: three ways, ranked

Q3 asks you to flatten `[[101,102],[103,104],[105]]`. There are three standard answers and knowing all three lets you pick the right one under pressure: the **loop** (clearest to explain), the **comprehension** (shortest, expected in most exams) and **`itertools.chain`** (fastest and cleanest for big data).

In [5]:
orders = [[101, 102], [103, 104], [105]]

# 1. THE LOOP — most readable, easiest to defend verbally
flat_loop = []
for group in orders:          # group is one inner list, e.g. [101, 102]
    for order in group:       # order is one number inside it
        flat_loop.append(order)

# 2. THE COMPREHENSION — read the two `for`s in the SAME left-to-right order as the loop above
flat_comp = [order for group in orders for order in group]

# 3. itertools.chain — the professional choice; doesn't build intermediate lists
from itertools import chain
flat_chain = list(chain.from_iterable(orders))

print("loop       :", flat_loop)
print("comprehension:", flat_comp)
print("chain      :", flat_chain)

# 🔧 CHANGE THIS: for a 3-level nest you need a recursive helper instead —
# see 06_functions.ipynb Q17 for the recursive `flatten()`.

loop       : [101, 102, 103, 104, 105]
comprehension: [101, 102, 103, 104, 105]
chain      : [101, 102, 103, 104, 105]


### ⚡ Beyond the syllabus: `namedtuple`: a tuple whose fields have names

The weakness of a plain tuple is that `branch[1]` tells the reader nothing. A `namedtuple` keeps every advantage of a tuple (fast, frozen, unpackable) but lets you write `branch.revenue`. It's the cheapest readability upgrade in Python and marks out code that's been thought about.

In [6]:
from collections import namedtuple

Branch = namedtuple('Branch', ['city', 'revenue', 'headcount'])   # define the shape once

delhi  = Branch('Delhi', 250000, 42)
mumbai = Branch('Mumbai', 410000, 65)

print(delhi.city, "earned ₹", f"{delhi.revenue:,}")   # readable — no mystery [0] / [1]
print("Still a real tuple, so unpacking works:")
city, revenue, headcount = mumbai
print(f"  {city} | ₹{revenue:,} | {headcount} staff")

# and it still behaves like a tuple everywhere else
network = [delhi, mumbai]
print("Total revenue:", f"₹{sum(b.revenue for b in network):,}")
print("Busiest branch:", max(network, key=lambda b: b.revenue).city)

Delhi earned ₹ 250,000
Still a real tuple, so unpacking works:
  Mumbai | ₹410,000 | 65 staff
Total revenue: ₹660,000
Busiest branch: Mumbai


## Part 2: Nested dictionaries

Same idea, but you step in by *name* rather than by position: `employee['details']['city']`. Because the keys are words, this stays readable no matter how deep it goes, which is exactly why JSON and API responses are built this way.

**Q4. How can you access the 'city' of employee in dictionary `employee={'id':101,'details':{'name':'Anita','city':'Pune'}}`??**

In [7]:
# Solution for Q4
employee = {'id': 101, 'details': {'name': 'Anita', 'city': 'Pune'}}
city = employee['details']['city']  # chained key access into nested dict
print(f"Employee city: {city}")

Employee city: Pune


**Q5. How can you add a new key 'department':'HR' under 'details' in dictionary `employee={'id':101,'details':{'name':'Anita','city':'Pune'}}`??**

In [8]:
# Solution for Q5
employee = {'id': 101, 'details': {'name': 'Anita', 'city': 'Pune'}}
employee['details']['department'] = 'HR'  # add nested key-value pair
print(f"Employee with department: {employee}")

Employee with department: {'id': 101, 'details': {'name': 'Anita', 'city': 'Pune', 'department': 'HR'}}


**Q6. How can you update the 'city' from 'Pune' to 'Mumbai' in nested dictionary `employee={'id':101,'details':{'name':'Anita','city':'Pune'}}`??**

In [9]:
# Solution for Q6
employee = {'id': 101, 'details': {'name': 'Anita', 'city': 'Pune'}}
employee['details']['city'] = 'Mumbai'  # update nested key directly
print(f"Relocated employee: {employee}")

Relocated employee: {'id': 101, 'details': {'name': 'Anita', 'city': 'Mumbai'}}


## Part 3: Tuples

A tuple looks like a list but uses round brackets and **cannot be changed after creation**. That sounds like a limitation; it's actually a guarantee. Use a tuple whenever a group of values belongs together permanently: a coordinate, a database row, a (branch, revenue) pair.

The killer feature is **unpacking**: `city, revenue = branch` splits the tuple across two named variables in one line.

**Q7. Given a tuple of stock prices `prices=(250,275,290,310)`, how can you access the last price??**

In [10]:
# Solution for Q7
prices = (250, 275, 290, 310)
last_price = prices[-1]  # negative index fetches final entry
print(f"Latest stock price: ₹{last_price}")

Latest stock price: ₹310


**Q8. How can you unpack the tuple `branch=('Delhi',250000)` into two separate variables??**

In [11]:
# Solution for Q8
branch = ('Delhi', 250000)
city, revenue = branch  # tuple unpacking
print(f"Branch city: {city}\nQuarterly revenue: ₹{revenue}")

Branch city: Delhi
Quarterly revenue: ₹250000


## Part 4: Sets

A set holds **unique** values only, in no particular order. Adding a duplicate does nothing, which makes `set(my_list)` the fastest way to de-duplicate anything. The real power is the overlap operators:

| Question | Operator |
|---|---|
| Who is in **either** list? | `A \| B` (union) |
| Who is in **both**? | `A & B` (intersection) |
| Who is in A but **not** B? | `A - B` (difference) |
| Who is in exactly one? | `A ^ B` (symmetric difference) |

**Q9. Given two sets of customers `setA={'Ravi','Meera','John'}` and `setB={'John','Ali','Kiran'}`, how can you find the common customers??**

In [12]:
# Solution for Q9
setA = {'Ravi', 'Meera', 'John'}
setB = {'John', 'Ali', 'Kiran'}
common_customers = setA.intersection(setB)  # overlap of both channels
print(f"Shared customers: {common_customers}")

Shared customers: {'John'}


**Q10. How can you add a new supplier 'GlobalCorp' to the set `suppliers={'TCS','Infosys','Wipro'}`??**

In [13]:
# Solution for Q10
suppliers = {'TCS', 'Infosys', 'Wipro'}
suppliers.add('GlobalCorp')  # sets support add for single elements
print(f"Supplier pool: {suppliers}")

Supplier pool: {'Wipro', 'GlobalCorp', 'Infosys', 'TCS'}


### ⚡ Beyond the syllabus: set algebra answers churn questions in one line

Retention analysis is pure set logic. Once last month's and this month's customer lists are sets, every business question becomes a single operator, with no loops and no counters, and it runs instantly even on millions of IDs.

In [14]:
last_month = {'Ravi', 'Meera', 'John', 'Ali', 'Kiran'}
this_month = {'John', 'Ali', 'Kiran', 'Sneha', 'Dev'}

print("Retained (in both)      :", last_month & this_month)
print("Churned (lost)          :", last_month - this_month)
print("Newly acquired          :", this_month - last_month)
print("Everyone we ever served :", last_month | this_month)
print("Moved in one direction  :", last_month ^ this_month)   # churned + new, i.e. everything that changed

# Turn it into the metrics a manager actually asks for
retention = len(last_month & this_month) / len(last_month) * 100
churn     = len(last_month - this_month) / len(last_month) * 100
print(f"\nRetention rate: {retention:.1f}%   |   Churn rate: {churn:.1f}%")

# De-duplicating a messy list — the most common everyday use of a set
raw = ['Ravi', 'ravi', 'RAVI', 'Meera', 'Ravi']
print("\nNaive set()          :", set(raw))                              # case differences survive
print("Normalised first     :", {name.strip().lower() for name in raw})  # set comprehension

Retained (in both)      : {'John', 'Kiran', 'Ali'}
Churned (lost)          : {'Meera', 'Ravi'}
Newly acquired          : {'Sneha', 'Dev'}
Everyone we ever served : {'Meera', 'Ravi', 'Sneha', 'Dev', 'Ali', 'Kiran', 'John'}
Moved in one direction  : {'Meera', 'Ravi', 'Sneha', 'Dev'}

Retention rate: 60.0%   |   Churn rate: 40.0%

Naive set()          : {'RAVI', 'Meera', 'Ravi', 'ravi'}
Normalised first     : {'meera', 'ravi'}


---

## Exam quick-reference

| To do this | Write this |
|---|---|
| 2nd inner list, 1st item | `sales[1][0]` |
| Change a nested value | `sales[1][1] = 1550` |
| Flatten one level | `[x for grp in nested for x in grp]` |
| Nested dict lookup | `emp['details']['city']` |
| Add a nested key | `emp['details']['dept'] = 'HR'` |
| Last item of a tuple | `prices[-1]` |
| Unpack a tuple | `city, revenue = branch` |
| Tuple with one item | `(5,)`, the comma is required |
| Common to both sets | `A & B` |
| In A only | `A - B` |
| Combine sets | `A | B` |
| Add to a set | `suppliers.add('GlobalCorp')` |
| De-duplicate a list | `list(set(lst))` |

### Adapting this in the exam

- Deeper nesting? Just add another bracket: `data[0][1][2]`. Print each step if unsure.
- The question says 'unique' or 'distinct'? That's your signal to reach for a set.
- The question says 'common', 'both', 'overlap', 'shared'? That's `&`.

### Traps that cost marks

- Tuples are **immutable**: `prices[0] = 300` raises `TypeError`. Rebuild the whole tuple instead.
- `(5)` is just the number 5. A one-item tuple needs the trailing comma: `(5,)`.
- Sets have **no order**, so `my_set[0]` is an error. Convert with `list(my_set)` first, and don't assume the order is stable.
- `set.add()` adds one item; `set.update()` adds many. Using `add` with a list raises `TypeError: unhashable type`.
- Flattening with a comprehension needs the `for` clauses in the *same order* as the nested loop; reversing them silently gives the wrong answer.